In [ ]:
import os
import pickle
import numpy as np
from tqdm.notebook import tqdm
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

In [ ]:
from ultralytics import YOLO

In [ ]:
BASE_DIR = 'E:\Desktop\capston\archive'
WORKING_DIR = 'E:\Desktop\capston'

In [ ]:
# Load VGG16 model
vgg_model = VGG16()
vgg_model = Model(inputs=vgg_model.inputs, outputs=vgg_model.layers[-2].output)
print(vgg_model.summary())


In [ ]:
# Load YOLO model
yolo_model = YOLO('yolov5s.pt')  # <--- Loaded YOLO model

In [ ]:
from sklearn.preprocessing import normalize  # For normalizing features
import numpy as np
from keras.preprocessing.image import load_img, img_to_array
from keras.applications.vgg16 import preprocess_input
import pickle
from tqdm import tqdm

# --- Modified Feature Extraction ---
features = {}
directory = 'E:\\Desktop\\capston\\archive\\Images'

# Process only a subset of images
subset_size = 100  # Adjust this value as needed
image_names = os.listdir(directory)[:subset_size]  # Get the first 'subset_size' images

# Fixed size for YOLO features
YOLO_FEATURE_SIZE = 2048

for img_name in tqdm(image_names):
    img_path = os.path.join(directory, img_name)
    
    try:
        # --- VGG16 Feature Extraction ---
        vgg_image = load_img(img_path, target_size=(224, 224))
        vgg_image = img_to_array(vgg_image)
        vgg_image = vgg_image.reshape((1, vgg_image.shape[0], vgg_image.shape[1], vgg_image.shape[2]))
        vgg_image = preprocess_input(vgg_image)
        vgg_feature = vgg_model.predict(vgg_image, verbose=0)  # Feature size: (4096,)

        # --- YOLO Feature Extraction ---
        yolo_results = yolo_model(img_path)  # Run YOLO on the image
        
        if yolo_results[0].boxes:
            boxes = yolo_results[0].boxes.xyxy.numpy()  # Bounding box coordinates
            confidences = yolo_results[0].boxes.conf.numpy()  # Confidence scores
            classes = yolo_results[0].boxes.cls.numpy()  # Class labels
            yolo_feature = np.concatenate([boxes.flatten(), confidences, classes])
        else:
            yolo_feature = np.zeros(YOLO_FEATURE_SIZE)
        
        if yolo_feature.shape[0] < YOLO_FEATURE_SIZE:
            yolo_feature = np.pad(yolo_feature, (0, YOLO_FEATURE_SIZE - yolo_feature.shape[0]), mode='constant')
        elif yolo_feature.shape[0] > YOLO_FEATURE_SIZE:
            yolo_feature = yolo_feature[:YOLO_FEATURE_SIZE]

        yolo_feature = normalize(yolo_feature.reshape(1, -1), norm='l2').flatten()
        combined_feature = np.concatenate((vgg_feature.flatten(), yolo_feature))
        
        if combined_feature.shape[0] == 6144:
            image_id = img_name.split('.')[0]
            features[image_id] = combined_feature
        else:
            print(f"Skipping {img_name}: Incorrect combined feature size {combined_feature.shape[0]}")

    except Exception as e:
        print(f"Error processing {img_name}: {e}")
        continue

pickle.dump(features, open(os.path.join(WORKING_DIR, 'combined_features.pkl'), 'wb'))


In [ ]:
correct_count = 0
incorrect_count = 0

for key, feature in features.items():
    if feature.shape[0] == 6144:
        correct_count += 1
    else:
        incorrect_count += 1
        print(f"Key {key}: Incorrect feature size {feature.shape[0]}")

print(f"Correct features: {correct_count}")
print(f"Incorrect features: {incorrect_count}")


In [ ]:
# --- Process Captions ---
with open('E:\\Desktop\\capston\\archive\\captions.txt', 'r') as f:
    next(f)
    captions_doc = f.read()

mapping = {}
for line in tqdm(captions_doc.split('\n')):
    tokens = line.split(',')
    if len(line) < 2:
        continue
    image_id, caption = tokens[0], tokens[1:]
    image_id = image_id.split('.')[0]
    caption = " ".join(caption)
    if image_id not in mapping:
        mapping[image_id] = []
    mapping[image_id].append(caption)

def clean(mapping):
    for key, captions in mapping.items():
        for i in range(len(captions)):
            caption = captions[i].lower()
            caption = caption.replace('[^A-Za-z]', '')
            caption = caption.replace('\s+', ' ')
            caption = 'startseq ' + " ".join([word for word in caption.split() if len(word) > 1]) + ' endseq'
            captions[i] = caption

clean(mapping)
subset_image_ids = set(features.keys())
mapping = {key: captions for key, captions in mapping.items() if key in subset_image_ids}

all_captions = []
for key in mapping:
    for caption in mapping[key]:
        all_captions.append(caption)

tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(caption.split()) for caption in all_captions)


In [ ]:
image_ids = list(mapping.keys())
split = int(len(image_ids) * 0.90)
train = image_ids[:split]
test = image_ids[split:]


In [ ]:
import time
import numpy as np

def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = list(), list(), list()
    n = 0
    while True:
        for key in data_keys:
            if key not in features or features[key].shape[0] != 6144:
                continue
            captions = mapping[key]
            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    X1.append(features[key])
                    X2.append(in_seq)
                    y.append(out_seq)
            n += 1
            if n == batch_size:
                yield {"image": np.array(X1), "text": np.array(X2)}, np.array(y)
                X1, X2, y = list(), list(), list()
                n = 0


In [ ]:
# --- Model Architecture ---
inputs1 = Input(shape=(6144,), name="image")
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)
inputs2 = Input(shape=(max_length,), name="text")
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)
model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')
plot_model(model, show_shapes=True, to_file='model_plot.png')


In [ ]:
batch_size = min(32, len(train))
steps = max(1, len(train) // batch_size)
for i in range(20):
    generator = data_generator(train, mapping, features, tokenizer, max_length, vocab_size, batch_size)
    model.fit(generator, epochs=1, steps_per_epoch=steps, verbose=1)
model.save(WORKING_DIR + '/best_model.h5')


In [ ]:
def idx_to_word(index, tokenizer):
    for word, i in tokenizer.word_index.items():
        if i == index:
            return word
    return None


In [ ]:
# --- Predict and Evaluate ---
def predict_caption(model, image, tokenizer, max_length):
    in_text = 'startseq'
    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length, padding='post')
        yhat = np.argmax(model.predict([image, sequence], verbose=0))
        word = idx_to_word(yhat, tokenizer)
        if word is None:
            break
        in_text += " " + word
        if word == 'endseq':
            break
    return in_text

from nltk.translate.bleu_score import corpus_bleu
actual, predicted = list(), list()
for key in tqdm(test):
    if key not in features or features[key].shape[0] != 6144:
        continue
    y_pred = predict_caption(model, features[key].reshape(1, -1), tokenizer, max_length).split()
    actual.append([caption.split() for caption in mapping[key]])
    predicted.append(y_pred)
print("BLEU-1: %f" % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
print("BLEU-2: %f" % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))
